In [40]:
from pathlib import Path
import pandas as pd

In [41]:
# Project paths
BASE_DIR = Path.cwd().parent

SPENDING_FILE = BASE_DIR / "data/raw/spending_profile.csv"
CARDS_FILE = BASE_DIR / "data/raw/credit_cards.csv"
REPORT_DIR = BASE_DIR / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Required columns
SPENDING_COLUMNS = ["category", "monthly_spending"]

CARD_COLUMNS = [
    "card_name",
    "annual_fee",
    "reward_type",
    "groceries_rate",
    "dining_rate",
    "gas_rate",
    "travel_rate",
    "bills_rate",
    "other_rate",
    "welcome_bonus_value",
    "source_url",
    "verified_date",
    "data_status"
]

RATE_COLUMNS = [
    "groceries_rate",
    "dining_rate",
    "gas_rate",
    "travel_rate",
    "bills_rate",
    "other_rate"
]


def check_required_columns(df, required_columns, dataset_name):
    missing_columns = [
        col for col in required_columns if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: {missing_columns}"
        )


def main():

    # Load datasets
    spending = pd.read_csv(SPENDING_FILE)
    cards = pd.read_csv(CARDS_FILE)

    check_required_columns(
        spending, SPENDING_COLUMNS, "Spending dataset"
    )

    check_required_columns(
        cards, CARD_COLUMNS, "Credit card dataset"
    )

    # Clean text columns
    spending["category"] = spending["category"].astype("string").str.strip()
    cards["card_name"] = cards["card_name"].astype("string").str.strip()
    cards["reward_type"] = cards["reward_type"].astype("string").str.strip()
    cards["data_status"] = cards["data_status"].astype("string").str.strip()

    # Convert numeric columns
    spending["monthly_spending"] = pd.to_numeric(
        spending["monthly_spending"], errors="coerce"
    )

    numeric_card_columns = [
        "annual_fee",
        "welcome_bonus_value"
    ] + RATE_COLUMNS

    for col in numeric_card_columns:
        cards[col] = pd.to_numeric(cards[col], errors="coerce")

    # Basic spending checks
    spending_duplicates = int(spending.duplicated().sum())
    missing_spending_categories = int(spending["category"].isna().sum())
    missing_spending_amounts = int(
        spending["monthly_spending"].isna().sum()
    )
    negative_spending = int(
        (spending["monthly_spending"] < 0).sum()
    )

    # Basic card checks
    card_duplicates = int(cards.duplicated().sum())
    missing_card_names = int(cards["card_name"].isna().sum())
    missing_fees = int(cards["annual_fee"].isna().sum())

    negative_fees = int((cards["annual_fee"] < 0).sum())

    missing_rates = int(cards[RATE_COLUMNS].isna().sum().sum())

    invalid_rates = int(
        (
            (cards[RATE_COLUMNS] < 0) |
            (cards[RATE_COLUMNS] > 1)
        ).sum().sum()
    )

    negative_bonuses = int(
        (cards["welcome_bonus_value"] < 0).sum()
    )

    missing_sources = int(
        cards["source_url"].isna().sum()
    )

    unverified_cards = int(
        (cards["data_status"].str.upper() != "VERIFIED").sum()
    )

    # Spending summary
    total_monthly_spending = spending["monthly_spending"].sum()
    total_annual_spending = total_monthly_spending * 12

    # Build quality report
    quality_summary = pd.DataFrame([
        ["Spending rows", len(spending)],
        ["Spending columns", len(spending.columns)],
        ["Card rows", len(cards)],
        ["Card columns", len(cards.columns)],
        ["Duplicate spending rows", spending_duplicates],
        ["Duplicate card rows", card_duplicates],
        ["Missing spending categories", missing_spending_categories],
        ["Missing spending amounts", missing_spending_amounts],
        ["Negative spending amounts", negative_spending],
        ["Missing card names", missing_card_names],
        ["Missing annual fees", missing_fees],
        ["Negative annual fees", negative_fees],
        ["Missing reward rates", missing_rates],
        ["Invalid reward rates", invalid_rates],
        ["Negative welcome bonuses", negative_bonuses],
        ["Cards without source URLs", missing_sources],
        ["Cards not marked VERIFIED", unverified_cards],
        ["Total monthly spending USD", total_monthly_spending],
        ["Estimated annual spending USD", total_annual_spending]
    ], columns=["metric", "value"])

    # Detailed category report
    spending_report = pd.DataFrame({
        "category": spending["category"],
        "monthly_spending": spending["monthly_spending"],
        "annual_spending": spending["monthly_spending"] * 12
    })

    # Detailed card report
    card_report = cards.copy()
    card_report["source_missing"] = card_report["source_url"].isna()
    card_report["verification_status"] = (
        card_report["data_status"].fillna("MISSING").str.upper()
    )

    # Save reports
    quality_summary.to_csv(
        REPORT_DIR / "data_quality_summary.csv", index=False
    )

    spending_report.to_csv(
        REPORT_DIR / "spending_quality_report.csv", index=False
    )

    card_report.to_csv(
        REPORT_DIR / "card_quality_report.csv", index=False
    )

    # Display results
    print("\nCREDIT CARD REWARDS OPTIMIZER")
    print("DATA QUALITY REPORT")
    print("-" * 45)

    print(quality_summary.to_string(index=False))

    print("\nReports saved successfully in the reports folder.")


if __name__ == "__main__":
    main()


CREDIT CARD REWARDS OPTIMIZER
DATA QUALITY REPORT
---------------------------------------------
                       metric  value
                Spending rows      6
             Spending columns      2
                    Card rows      3
                 Card columns     13
      Duplicate spending rows      0
          Duplicate card rows      0
  Missing spending categories      0
     Missing spending amounts      0
    Negative spending amounts      0
           Missing card names      0
          Missing annual fees      0
         Negative annual fees      0
         Missing reward rates      0
         Invalid reward rates      0
     Negative welcome bonuses      0
    Cards without source URLs      0
    Cards not marked VERIFIED      0
   Total monthly spending USD   1830
Estimated annual spending USD  21960

Reports saved successfully in the reports folder.
